# fhir Notebook

## 

### Fetching source data to RAW folder 

01_ingest_raw

In [14]:
def fetch_all_pages(resource_type, count=20, since_date=None, max_pages=10):
    extra_params = "&_sort=-_lastUpdated"
    if since_date:
        extra_params += f"&_lastUpdated=ge{since_date}"
    
    url = f"https://hapi.fhir.org/baseR4/{resource_type}?_count={count}{extra_params}"
    all_pages = []
    page_count = 0

    while url and page_count < max_pages:
        resp = requests.get(url)
        resp.raise_for_status()
        page_data = resp.json()
        all_pages.append(page_data)
        page_count += 1

        next_url = None
        for link in page_data.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")
                break
        url = next_url

    print(f"Stopped after {page_count} pages")
    return all_pages

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 15, Finished, Available, Finished, False)

In [15]:
pages = fetch_all_pages("Patient", count=50, max_pages=5)
print(f"Fetched {len(pages)} pages")
print(f"First page has {len(pages[0].get('entry', []))} entries")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 16, Finished, Available, Finished, False)

Stopped after 5 pages
Fetched 5 pages
First page has 50 entries


In [16]:
import json, os
from datetime import datetime

def save_raw_pages(resource_type, pages, since_date=None):
    extraction_ts = datetime.utcnow().isoformat()
    extraction_date = datetime.utcnow().strftime("%Y-%m-%d")
    folder_path = f"/lakehouse/default/Files/raw/{resource_type}/{extraction_date}"
    os.makedirs(folder_path, exist_ok=True)

    for i, page in enumerate(pages):
        record = {
            "extraction_timestamp": extraction_ts,
            "api_url_or_params": f"https://hapi.fhir.org/baseR4/{resource_type}",
            "page_number": i,
            "data": page
        }
        with open(f"{folder_path}/page_{i}.json", "w") as f:
            json.dump(record, f)

    print(f"Saved {len(pages)} pages to {folder_path}")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 17, Finished, Available, Finished, False)

In [17]:
save_raw_pages("Patient", pages)

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 18, Finished, Available, Finished, False)

Saved 5 pages to /lakehouse/default/Files/raw/Patient/2026-07-25


In [18]:
for resource in ["Encounter", "Observation", "Condition"]:
    pages = fetch_all_pages(resource, count=50, max_pages=5)
    save_raw_pages(resource, pages)
    print(f"{resource} done\n")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 19, Finished, Available, Finished, False)

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Encounter/2026-07-25
Encounter done

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Observation/2026-07-25
Observation done

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Condition/2026-07-25
Condition done



## Bronze layer 

### Patient Bronze table

In [19]:
from pyspark.sql import functions as F

raw_df = spark.read.option("multiline", "true").json("Files/raw/Patient/*/*.json")
raw_df.printSchema()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 20, Finished, Available, Finished, False)

root
 |-- api_url_or_params: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- entry: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- fullUrl: string (nullable = true)
 |    |    |    |-- resource: struct (nullable = true)
 |    |    |    |    |-- _birthDate: struct (nullable = true)
 |    |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |    |-- address: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- city: string (nullable = true)
 |    |    |    |    |    |    |-- country: string (nullable = true)
 |    |    |    |    |    |    |-- di

In [20]:
exploded_df = raw_df.select(
    "extraction_timestamp",
    "api_url_or_params",
    F.explode("data.entry").alias("entry")
)

bronze_patient = exploded_df.select(
    F.col("entry.resource.id").alias("patient_id"),
    F.col("entry.resource.gender").alias("gender"),
    F.col("entry.resource.birthDate").alias("birth_date"),
    F.col("entry.resource.name")[0]["family"].alias("last_name"),
    F.col("entry.resource.name")[0]["given"][0].alias("first_name"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp",
    "api_url_or_params"
)

display(bronze_patient.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9fabcd6-bfa8-4ce7-a433-4c5e61e4503d)

In [21]:
bronze_patient.write.format("delta").mode("append").saveAsTable("bronze_patient")
print("bronze_patient table created")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 22, Finished, Available, Finished, False)

bronze_patient table created


In [22]:
spark.sql("SELECT * FROM bronze_patient LIMIT 5").show()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 23, Finished, Available, Finished, False)

+----------+------+----------+---------+----------+--------------------+--------------------+--------------------+
|patient_id|gender|birth_date|last_name|first_name|  full_resource_json|extraction_timestamp|   api_url_or_params|
+----------+------+----------+---------+----------+--------------------+--------------------+--------------------+
| 137222841|female|1962-04-12| Thompson|     Sarah|{"birthDate":"196...|2026-07-25T11:44:...|https://hapi.fhir...|
| 137222839|female|1962-04-12| Thompson|     Sarah|{"birthDate":"196...|2026-07-25T11:44:...|https://hapi.fhir...|
| 137222836|female|1962-04-12| Thompson|     Sarah|{"birthDate":"196...|2026-07-25T11:44:...|https://hapi.fhir...|
| 137222837|female|1962-04-12| Thompson|     Sarah|{"birthDate":"196...|2026-07-25T11:44:...|https://hapi.fhir...|
| 137222834|female|1962-04-12| Thompson|     Sarah|{"birthDate":"196...|2026-07-25T11:44:...|https://hapi.fhir...|
+----------+------+----------+---------+----------+--------------------+--------

#### Encounter Bronze table

In [23]:
raw_df_enc = spark.read.option("multiline", "true").json("Files/raw/Encounter/*/*.json")
raw_df_enc.printSchema()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 24, Finished, Available, Finished, False)

root
 |-- api_url_or_params: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- entry: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- fullUrl: string (nullable = true)
 |    |    |    |-- resource: struct (nullable = true)
 |    |    |    |    |-- appointment: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- reference: string (nullable = true)
 |    |    |    |    |-- class: struct (nullable = true)
 |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |-- diagnosis: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- condition: struct (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nulla

In [24]:
exploded_df_enc = raw_df_enc.select(
    "extraction_timestamp",
    "api_url_or_params",
    F.explode("data.entry").alias("entry")
)

bronze_encounter = exploded_df_enc.select(
    F.col("entry.resource.id").alias("encounter_id"),
    F.col("entry.resource.status").alias("status"),
    F.col("entry.resource.class.code").alias("class_code"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp",
    "api_url_or_params"
)

display(bronze_encounter.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81210e55-9942-4520-afa2-f4b843cd53ec)

In [25]:
bronze_encounter.write.format("delta").mode("append").saveAsTable("bronze_encounter")
print("bronze_encounter table created")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 26, Finished, Available, Finished, False)

bronze_encounter table created


In [26]:
spark.sql("SELECT * FROM bronze_encounter LIMIT 5").show()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 27, Finished, Available, Finished, False)

+--------------------+--------+----------+--------------------+--------------------+--------------------+--------------------+
|        encounter_id|  status|class_code|         patient_ref|  full_resource_json|extraction_timestamp|   api_url_or_params|
+--------------------+--------+----------+--------------------+--------------------+--------------------+--------------------+
|3545148b-de09-45e...| arrived|       AMB|Patient/6cfcb3db-...|{"appointment":[{...|2026-07-25T11:45:...|https://hapi.fhir...|
|d2c5e813-dc29-499...| arrived|       AMB|Patient/bb708418-...|{"appointment":[{...|2026-07-25T11:45:...|https://hapi.fhir...|
|4c198e4b-a868-460...|finished|       AMB|Patient/e9756c82-...|{"appointment":[{...|2026-07-25T11:45:...|https://hapi.fhir...|
|407c9e7c-d3a1-410...| arrived|       AMB|Patient/bb708418-...|{"appointment":[{...|2026-07-25T11:45:...|https://hapi.fhir...|
|85f47f48-a8cc-4aa...|finished|       AMB|Patient/63206594-...|{"appointment":[{...|2026-07-25T11:45:...|https:

#### Observation table

In [27]:
raw_df_obs = spark.read.option("multiline", "true").json("Files/raw/Observation/*/*.json")
raw_df_obs.printSchema()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 28, Finished, Available, Finished, False)

root
 |-- api_url_or_params: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- entry: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- fullUrl: string (nullable = true)
 |    |    |    |-- resource: struct (nullable = true)
 |    |    |    |    |-- basedOn: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |-- identifier: struct (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |-- category: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |  

In [28]:
exploded_df_obs = raw_df_obs.select(
    "extraction_timestamp",
    "api_url_or_params",
    F.explode("data.entry").alias("entry")
)

bronze_observation = exploded_df_obs.select(
    F.col("entry.resource.id").alias("observation_id"),
    F.col("entry.resource.status").alias("status"),
    F.col("entry.resource.code.coding")[0]["display"].alias("obs_type"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.col("entry.resource.encounter.reference").alias("encounter_ref"),
    F.col("entry.resource.effectiveDateTime").alias("effective_datetime"),
    F.col("entry.resource.valueQuantity.value").alias("value"),
    F.col("entry.resource.valueQuantity.unit").alias("unit"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp",
    "api_url_or_params"
)

display(bronze_observation.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 082dca93-2423-472b-9784-95df495d2339)

In [29]:
bronze_observation.write.format("delta").mode("append").saveAsTable("bronze_observation")
print("bronze_observation table created")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 30, Finished, Available, Finished, False)

bronze_observation table created


### 

In [32]:
spark.sql("SELECT * FROM bronze_observation LIMIT 5").show()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 33, Finished, Available, Finished, False)

+--------------+------+--------+-----------------+-------------------+--------------------+-----+--------+--------------------+--------------------+--------------------+
|observation_id|status|obs_type|      patient_ref|      encounter_ref|  effective_datetime|value|    unit|  full_resource_json|extraction_timestamp|   api_url_or_params|
+--------------+------+--------+-----------------+-------------------+--------------------+-----+--------+--------------------+--------------------+--------------------+
|     137228177| final|    NULL|Patient/137228124|Encounter/137228166|2026-07-25T12:52:...| 79.5|      Kg|{"code":{"coding"...|2026-07-25T11:45:...|https://hapi.fhir...|
|     137228197| final|    NULL|Patient/137228124|Encounter/137228167|2026-06-27T12:52:...| 78.3|      Kg|{"code":{"coding"...|2026-07-25T11:45:...|https://hapi.fhir...|
|     137228202| final|    NULL|Patient/137228124|Encounter/137228167|2026-06-20T12:52:...| 74.9|      Kg|{"code":{"coding"...|2026-07-25T11:45:...|ht

#### Condition table

In [33]:
raw_df_cond = spark.read.option("multiline", "true").json("Files/raw/Condition/*/*.json")
raw_df_cond.printSchema()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 34, Finished, Available, Finished, False)

root
 |-- api_url_or_params: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- entry: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- fullUrl: string (nullable = true)
 |    |    |    |-- resource: struct (nullable = true)
 |    |    |    |    |-- asserter: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |    |-- valueCoding: struct (nullable = true)
 |    |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- identifier: struct (nullable = true)
 |    |    |    |    |    |    |-- system: string (nullable = true)
 |    | 

In [34]:
exploded_df_cond = raw_df_cond.select(
    "extraction_timestamp",
    "api_url_or_params",
    F.explode("data.entry").alias("entry")
)

bronze_condition = exploded_df_cond.select(
    F.col("entry.resource.id").alias("condition_id"),
    F.col("entry.resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),
    F.col("entry.resource.code.coding")[0]["display"].alias("condition_name"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.col("entry.resource.onsetDateTime").alias("onset_datetime"),
    F.col("entry.resource.recordedDate").alias("recorded_date"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp",
    "api_url_or_params"
)

display(bronze_condition.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 113a5092-1c1d-44d8-8b38-71cde9b8e90e)

In [35]:
bronze_condition.write.format("delta").mode("append").saveAsTable("bronze_condition")
print("bronze_condition table created")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 36, Finished, Available, Finished, False)

bronze_condition table created


In [36]:
spark.sql("SELECT * FROM bronze_condition LIMIT 5").show()

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 37, Finished, Available, Finished, False)

+------------+---------------+----------------+-----------------+--------------+-------------+--------------------+--------------------+--------------------+
|condition_id|clinical_status|  condition_name|      patient_ref|onset_datetime|recorded_date|  full_resource_json|extraction_timestamp|   api_url_or_params|
+------------+---------------+----------------+-----------------+--------------+-------------+--------------------+--------------------+--------------------+
|   137210806|         active|          angina|Patient/137210792|          NULL|         NULL|{"category":[{"co...|2026-07-25T11:45:...|https://hapi.fhir...|
|   137210800|         active|neuropathic pain|Patient/137210792|          NULL|         NULL|{"category":[{"co...|2026-07-25T11:45:...|https://hapi.fhir...|
|   137210803|         active|            GERD|Patient/137210792|          NULL|         NULL|{"category":[{"co...|2026-07-25T11:45:...|https://hapi.fhir...|
|   137210799|         active|      depression|Patie

## Silver layer

#### Silver Patient table

In [37]:
bronze_patient = spark.table("bronze_patient")

silver_patient_clean = bronze_patient.select(
    "patient_id",
    "gender",
    "birth_date",
    "last_name",
    "first_name",
    "extraction_timestamp"
).dropDuplicates(["patient_id", "extraction_timestamp"])

display(silver_patient_clean.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e06dd392-156d-41a3-8399-f0db3b03db10)

In [38]:
from delta.tables import DeltaTable

def scd2_merge(df_clean, table_name, business_key, tracked_cols):
    df_new = df_clean.withColumn(
        "row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in tracked_cols]), 256)
    ).withColumn("valid_from", F.col("extraction_timestamp")) \
     .withColumn("valid_to", F.lit(None).cast("string")) \
     .withColumn("is_current", F.lit(True))

    if not spark.catalog.tableExists(table_name):
        df_new.write.format("delta").saveAsTable(table_name)
        print(f"{table_name}: created, {df_new.count()} rows")
        return

    silver_table = DeltaTable.forName(spark, table_name)
    silver_table.alias("t").merge(
        df_new.alias("s"),
        f"t.{business_key} = s.{business_key} AND t.is_current = true"
    ).whenMatchedUpdate(
        condition="t.row_hash != s.row_hash",
        set={"is_current": "false", "valid_to": "s.extraction_timestamp"}
    ).execute()

    existing_current = spark.table(table_name).filter("is_current = true").select(business_key, "row_hash")
    to_insert = df_new.join(existing_current, [business_key, "row_hash"], "left_anti")
    to_insert.write.format("delta").mode("append").saveAsTable(table_name)
    print(f"{table_name}: inserted {to_insert.count()} new/changed rows")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 39, Finished, Available, Finished, False)

In [39]:
scd2_merge(silver_patient_clean, "silver_patient", "patient_id",
           ["gender", "birth_date", "last_name", "first_name"])

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 40, Finished, Available, Finished, False)

silver_patient: created, 250 rows


#### Silver Encounter table

In [40]:
bronze_encounter = spark.table("bronze_encounter")

silver_encounter_clean = bronze_encounter.select(
    "encounter_id",
    "status",
    "class_code",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    "extraction_timestamp"
).dropDuplicates(["encounter_id", "extraction_timestamp"])

display(silver_encounter_clean.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa7387ab-6f71-4097-a9be-60403ee512a2)

In [41]:
scd2_merge(silver_encounter_clean, "silver_encounter", "encounter_id",
           ["status", "class_code", "patient_id"])

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 42, Finished, Available, Finished, False)

silver_encounter: created, 250 rows


#### Silver Observation Table

In [42]:
bronze_observation = spark.table("bronze_observation")

silver_observation_clean = bronze_observation.select(
    "observation_id",
    "status",
    "obs_type",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    F.regexp_replace("encounter_ref", "Encounter/", "").alias("encounter_id"),
    "effective_datetime",
    "value",
    "unit",
    "extraction_timestamp"
).dropDuplicates(["observation_id", "extraction_timestamp"])

display(silver_observation_clean.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 97c012d3-a6c8-456e-b62d-0736ba57fd4f)

In [43]:
scd2_merge(silver_observation_clean, "silver_observation", "observation_id",
           ["status", "obs_type", "patient_id", "encounter_id", "value", "unit"])

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 44, Finished, Available, Finished, False)

silver_observation: created, 250 rows


#### Silver Condition table

In [44]:
bronze_condition = spark.table("bronze_condition")

silver_condition_clean = bronze_condition.select(
    "condition_id",
    "clinical_status",
    "condition_name",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    "onset_datetime",
    "recorded_date",
    "extraction_timestamp"
).dropDuplicates(["condition_id", "extraction_timestamp"])

display(silver_condition_clean.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2acfe53c-761c-493e-b352-1c2ea04edb67)

In [45]:
scd2_merge(silver_condition_clean, "silver_condition", "condition_id",
           ["clinical_status", "condition_name", "patient_id", "onset_datetime"])

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 46, Finished, Available, Finished, False)

silver_condition: created, 250 rows


## Gold Layer

#### Gold layer — first table (Patient + Encounter)

In [46]:
silver_patient = spark.table("silver_patient").filter("is_current = true")
silver_encounter = spark.table("silver_encounter").filter("is_current = true")

gold_patient_encounters = silver_patient.join(silver_encounter, "patient_id", "left").select(
    silver_patient.patient_id, "first_name", "last_name", "gender",
    "encounter_id", "status", "class_code"
)

gold_patient_encounters.write.format("delta").mode("overwrite").saveAsTable("gold_patient_encounter_summary")

display(gold_patient_encounters.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4be49081-049a-4e7f-8b17-776301368438)

#### Gold table 2 — Patient + Condition

In [47]:
silver_condition = spark.table("silver_condition").filter("is_current = true")

gold_patient_conditions = silver_patient.join(silver_condition, "patient_id", "left").select(
    silver_patient.patient_id, "first_name", "last_name",
    "condition_name", "clinical_status", "onset_datetime"
)

gold_patient_conditions.write.format("delta").mode("overwrite").saveAsTable("gold_patient_condition_summary")

display(gold_patient_conditions.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab18cdfb-b8a4-4b1d-8b51-3117f04f58cd)

#### Gold table 3 — Encounter + Observation

In [48]:
silver_observation = spark.table("silver_observation").filter("is_current = true")

gold_encounter_observations = silver_encounter.join(silver_observation, "encounter_id", "left").select(
    silver_encounter.encounter_id,
    silver_encounter.patient_id,
    silver_encounter.status,
    "obs_type", "value", "unit", "effective_datetime"
)

gold_encounter_observations.write.format("delta").mode("overwrite").saveAsTable("gold_encounter_observation_summary")

display(gold_encounter_observations.limit(5))

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8bafba1b-a6a8-4dfa-bf89-16d96ae34aaa)

## Incremental load

In [49]:
for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    pages = fetch_all_pages(resource, count=50, max_pages=5)
    save_raw_pages(resource, pages)
    print(f"{resource} re-ingested\n")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 50, Finished, Available, Finished, False)

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Patient/2026-07-25
Patient re-ingested

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Encounter/2026-07-25
Encounter re-ingested

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Observation/2026-07-25
Observation re-ingested

Stopped after 5 pages
Saved 5 pages to /lakehouse/default/Files/raw/Condition/2026-07-25
Condition re-ingested



#### Re running all 4 bronze 

In [50]:
# Patient
raw_df = spark.read.option("multiline", "true").json("Files/raw/Patient/*/*.json")
exploded_df = raw_df.select("extraction_timestamp", "api_url_or_params", F.explode("data.entry").alias("entry"))
bronze_patient = exploded_df.select(
    F.col("entry.resource.id").alias("patient_id"),
    F.col("entry.resource.gender").alias("gender"),
    F.col("entry.resource.birthDate").alias("birth_date"),
    F.col("entry.resource.name")[0]["family"].alias("last_name"),
    F.col("entry.resource.name")[0]["given"][0].alias("first_name"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp", "api_url_or_params"
)
bronze_patient.write.format("delta").mode("append").saveAsTable("bronze_patient")
print("bronze_patient updated")

# Encounter
raw_df_enc = spark.read.option("multiline", "true").json("Files/raw/Encounter/*/*.json")
exploded_df_enc = raw_df_enc.select("extraction_timestamp", "api_url_or_params", F.explode("data.entry").alias("entry"))
bronze_encounter = exploded_df_enc.select(
    F.col("entry.resource.id").alias("encounter_id"),
    F.col("entry.resource.status").alias("status"),
    F.col("entry.resource.class.code").alias("class_code"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp", "api_url_or_params"
)
bronze_encounter.write.format("delta").mode("append").saveAsTable("bronze_encounter")
print("bronze_encounter updated")

# Observation
raw_df_obs = spark.read.option("multiline", "true").json("Files/raw/Observation/*/*.json")
exploded_df_obs = raw_df_obs.select("extraction_timestamp", "api_url_or_params", F.explode("data.entry").alias("entry"))
bronze_observation = exploded_df_obs.select(
    F.col("entry.resource.id").alias("observation_id"),
    F.col("entry.resource.status").alias("status"),
    F.col("entry.resource.code.coding")[0]["display"].alias("obs_type"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.col("entry.resource.encounter.reference").alias("encounter_ref"),
    F.col("entry.resource.effectiveDateTime").alias("effective_datetime"),
    F.col("entry.resource.valueQuantity.value").alias("value"),
    F.col("entry.resource.valueQuantity.unit").alias("unit"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp", "api_url_or_params"
)
bronze_observation.write.format("delta").mode("append").saveAsTable("bronze_observation")
print("bronze_observation updated")

# Condition
raw_df_cond = spark.read.option("multiline", "true").json("Files/raw/Condition/*/*.json")
exploded_df_cond = raw_df_cond.select("extraction_timestamp", "api_url_or_params", F.explode("data.entry").alias("entry"))
bronze_condition = exploded_df_cond.select(
    F.col("entry.resource.id").alias("condition_id"),
    F.col("entry.resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),
    F.col("entry.resource.code.coding")[0]["display"].alias("condition_name"),
    F.col("entry.resource.subject.reference").alias("patient_ref"),
    F.col("entry.resource.onsetDateTime").alias("onset_datetime"),
    F.col("entry.resource.recordedDate").alias("recorded_date"),
    F.to_json(F.col("entry.resource")).alias("full_resource_json"),
    "extraction_timestamp", "api_url_or_params"
)
bronze_condition.write.format("delta").mode("append").saveAsTable("bronze_condition")
print("bronze_condition updated")

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 51, Finished, Available, Finished, False)

bronze_patient updated
bronze_encounter updated
bronze_observation updated
bronze_condition updated


#### Re run Silver for SCD Type2

In [51]:
# Patient
bronze_patient = spark.table("bronze_patient")
silver_patient_clean = bronze_patient.select(
    "patient_id", "gender", "birth_date", "last_name", "first_name", "extraction_timestamp"
).dropDuplicates(["patient_id", "extraction_timestamp"])
scd2_merge(silver_patient_clean, "silver_patient", "patient_id",
           ["gender", "birth_date", "last_name", "first_name"])

# Encounter
bronze_encounter = spark.table("bronze_encounter")
silver_encounter_clean = bronze_encounter.select(
    "encounter_id", "status", "class_code",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    "extraction_timestamp"
).dropDuplicates(["encounter_id", "extraction_timestamp"])
scd2_merge(silver_encounter_clean, "silver_encounter", "encounter_id",
           ["status", "class_code", "patient_id"])

# Observation
bronze_observation = spark.table("bronze_observation")
silver_observation_clean = bronze_observation.select(
    "observation_id", "status", "obs_type",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    F.regexp_replace("encounter_ref", "Encounter/", "").alias("encounter_id"),
    "effective_datetime", "value", "unit", "extraction_timestamp"
).dropDuplicates(["observation_id", "extraction_timestamp"])
scd2_merge(silver_observation_clean, "silver_observation", "observation_id",
           ["status", "obs_type", "patient_id", "encounter_id", "value", "unit"])

# Condition
bronze_condition = spark.table("bronze_condition")
silver_condition_clean = bronze_condition.select(
    "condition_id", "clinical_status", "condition_name",
    F.regexp_replace("patient_ref", "Patient/", "").alias("patient_id"),
    "onset_datetime", "recorded_date", "extraction_timestamp"
).dropDuplicates(["condition_id", "extraction_timestamp"])
scd2_merge(silver_condition_clean, "silver_condition", "condition_id",
           ["clinical_status", "condition_name", "patient_id", "onset_datetime"])

StatementMeta(, f57177c6-5f5c-4dd5-bad8-994371b1db94, 52, Finished, Available, Finished, False)

silver_patient: inserted 0 new/changed rows
silver_encounter: inserted 0 new/changed rows
silver_observation: inserted 0 new/changed rows
silver_condition: inserted 0 new/changed rows
